# Create tabular datasets on demand

The goal of this project is to generate realistic synthetic tabular datasets on demand using, served through a simple Gradio interface.

In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [2]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gradio as gr

In [3]:
# Constants

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [4]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [5]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [7]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="meta-llama/Llama-3.2-3B-Instruct")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

LICENSE.txt: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

USE_POLICY.md: 0.00B [00:00, ?B/s]

orig_params.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

original/tokenizer.model:   0%|          | 0.00/2.18M [00:00<?, ?B/s]

'/root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B-Instruct/snapshots/0cb88a4f764b7a12671c53f0838cd831a0843b95'

In [12]:
def generate_dataset(domain, rows, columns):
    rows = int(rows)
    column_list = [c.strip() for c in columns.split(",") if c.strip()]
    columns_text = "\n".join(column_list)

    system_message = """
    You are a helpful assistant that generates realistic synthetic datasets.
    Return only CSV. Do not include any explanation, preamble, or markdown formatting.
    """

    user_prompt = f"""
    Generate {rows} synthetic {domain} records.

    Columns:
    {columns_text}

    Return only CSV.
    """

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,   # tells the model it's its turn to respond
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True)  # cleaner console logs

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        streamer=streamer,
        pad_token_id=tokenizer.eos_token_id
    )

    input_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(
        outputs[0][input_length:],   # only the newly generated tokens
        skip_special_tokens=True
    )

    return response.strip()

In [13]:
import gradio as gr

demo = gr.Interface(
    fn=generate_dataset,
    inputs=[
        gr.Textbox(
            label="Dataset Domain",
            placeholder="Example: Healthcare"
        ),
        gr.Number(
            label="Number of Rows",
            value=10
        ),
        gr.Textbox(
            label="Columns (comma-separated)",
            placeholder="Example: ID, Name, Age, Diagnosis, Admission Date",
            value="ID, Name, Age"
        )
    ],
    outputs=gr.Textbox(
        label="Generated Dataset"
    ),
    title="Synthetic Dataset Generator",
    description="Generate synthetic datasets using Llama"
)

In [14]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://21d85977d02841afc0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
